In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from pprint import pformat

In [ ]:
from hloc import (
    helper_func_autarc,
    extract_features,
    match_features,
    pairs_from_exhaustive,
    pairs_from_covisibility,
    pairs_from_poses,
    reconstruction,
)

In [ ]:
import pycolmap

In [ ]:
# First Run use COLMAP
experiment = "FriedrichsHouse"
dataset = Path(f"/data/")  # change this if your dataset is somewhere else
images = dataset / experiment / "full"

outputs = Path(f"/data/output/hloc/{experiment}")  # where everything will be saved
outputs.mkdir(parents=True, exist_ok=True)

ref_sfm_pairs = outputs / "pairs-exhaustive.txt"
ref_sfm_dir = outputs / "sfm_sift+vocabtree"
ref_sfm_dir.mkdir(parents=True, exist_ok=True)
database_path = ref_sfm_dir / "database.db"


max_image_size = 1600
max_num_features = 2000
prior_position_std = 2.0
prior_position_z_std = 3.0

In [ ]:
pycolmap.extract_features(database_path, 
                          images,camera_model= "SIMPLE_RADIAL",  
                          device="cuda",
                          extraction_options={"max_image_size": max_image_size, 
                                              "use_gpu": True, 
                                              "sift":{"max_num_features": max_num_features}},                     
                        )

Matching

In [ ]:
import subprocess

command = [
    "colmap", "spatial_matcher",
    "--database_path", str(database_path)
]

result = subprocess.run(command, capture_output=True, text=True)
print("stdout:", result.stdout)
print("stderr:", result.stderr)
print("returncode:", result.returncode)

Pycolmap alternative

In [ ]:
""" pycolmap.match_spatial( database_path,
                        matching_options={"use_gpu": True,
                                         "max_num_matches": 32768,
                                         "sift": {
                                              "max_ratio": 0.8, 
                                              "max_distance": 0.7, 
                                              "cross_check": True, 
                                              "cpu_brute_force_matcher": False        
                                          },
                                        },
                        device="cuda"
                    ) """

Reconstruction 

In [ ]:
""" import subprocess

command = [
    "colmap", "pose_prior_mapper",
    "--database_path", str(database_path),
    "--image_path", str(images),
    "--output_path", str(sfm_dir),
    "--prior_position_std_x", str(prior_position_std),
    "--prior_position_std_y", str(prior_position_std),
    "--prior_position_std_z", str(prior_position_z_std),
    "--overwrite_priors_covariance", "1"
]

result = subprocess.run(command, capture_output=True, text=True)
print("stdout:", result.stdout)
print("stderr:", result.stderr)
print("returncode:", result.returncode) """

In [ ]:
# Move model 
""" largest_index = None
    largest_num_images = 0
    for index, rec in reconstructions.items():
        num_images = rec.num_reg_images()
        if num_images > largest_num_images:
            largest_index = index
            largest_num_images = num_images
    assert largest_index is not None
    logger.info(
        f"Largest model is #{largest_index} " f"with {largest_num_images} images."
    )

    for filename in [
        "images.bin",
        "cameras.bin",
        "points3D.bin",
        "frames.bin",
        "rigs.bin",
    ]:
        if (sfm_dir / filename).exists():
            (sfm_dir / filename).unlink()
        shutil.move(str(models_path / str(largest_index) / filename), str(sfm_dir))
    return reconstructions[largest_index] """

Pycolmap (not fully exposed yet)

In [ ]:
""" maps = pycolmap.incremental_mapping(database_path, images, sfm_dir, options={
    "use_prior_position": True,
    "prior_position_std": 2.0,
    "prior_position_z_std": 3.0, 
    "min_num_inliers": 15})
model = maps[0]  # get the first (and only) reconstruction """

In [ ]:
ref_model = pycolmap.Reconstruction()
ref_model.read(ref_sfm_dir / "0")

In [ ]:
ref_model.export_PLY(ref_sfm_dir / "model.ply")

Get Bounding Box for Masking

In [ ]:
from hloc import geofencing

bbox_min, bbox_max,  = geofencing.compute_adaptive_geofence(ref_model, silo_ratio=0.4, height_center_bias=0.8, safety_margin=0.1)

In [ ]:
from hloc import masking_geo

ret = masking_geo.create_mvs_masks(
    model= ref_model,
    output_mask_folder = images / "masks_geo",
    bbox_min=bbox_min,
    bbox_max=bbox_max,
)

## Setup Run 2

In [ ]:
experiment = "FriedrichsHouse"
dataset = Path(f"/data/")  # change this if your dataset is somewhere else
images = dataset / experiment / "full"

extractor = "superpoint_max"
matcher = "lightglue"
pairs = "pose" # covisibility, pose, exhaustive

outputs = Path(f"/data/output/hloc/{experiment}")  # where everything will be saved
outputs.mkdir(parents=True, exist_ok=True)

mvs_path = outputs / f"mvs_sfm_{extractor}+{matcher}"
mvs_path.mkdir(parents=True, exist_ok=True)
mvs_type = "PMVS"  # choose between "PMVS" and "COLMAP"

sfm_dir = outputs / f"sfm_{extractor}+{matcher}"
sfm_pairs = sfm_dir / f"pairs-{pairs}.txt"
sfm_dir.mkdir(parents=True, exist_ok=True)
database_path = sfm_dir / "database.db"

# retrieval_conf = extract_features.confs["netvlad"]
feature_conf = extract_features.confs[extractor]
# matcher_conf = match_features.confs[f"{extractor}+{matcher}"]
# matcher_conf = match_features.confs[f"{matcher}"]
matcher_conf = match_features.confs["superpoint+lightglue"]
# list the standard configurations available
# print(f"Configs for feature extractors:\n{pformat(extract_features.confs)}")
# print(f"Configs for feature matchers:\n{pformat(match_features.confs)}")

resize = True
max_image_size = 1600
# max_num_features = 2000
prior_position_std = 2.0
prior_position_z_std = 3.0

Downsaple Images

In [ ]:
from hloc.extract_features import resize_image 
from hloc.utils.io import read_image
import cv2
import os
from PIL import Image
import shutil

resized_images = images / "resized"
resized_images.mkdir(parents=True, exist_ok=True)   

# Resize all images
# Crates a list of image names in images (dir) without subdirs
images_list = [f.name for f in images.iterdir() if f.is_file()]

sample_image_name = os.listdir(images)[0]
img = Image.open(images / sample_image_name)
size = img.size  # (width, height)

size_new = size
if max_image_size and (
            resize or max(size) > max_image_size
        ):
            scale = max_image_size / max(size)
            size_new = tuple(int(round(x * scale)) for x in size)
            resize = True

if resize:
    for img_name in images_list:
        img_path = images / img_name
        img = read_image(img_path, grayscale=False)
        img = resize_image(img, size_new, "cv2_area")
        cv2.imwrite(str(resized_images / img_name), img[:, :, ::-1])
    # check if masks exist 
    
    if (images / "masks_geo").exists():
        resized_masks = images / "resized" / "masks_geo"
        resized_masks.mkdir(parents=True, exist_ok=True)  
        sample_mask_name = os.listdir(images / "masks_geo")[0]
        mask = Image.open(images / "masks_geo" / sample_mask_name)
        size = mask.size  # (width, height) 
        if size != size_new:
            print(f"Resizing masks from {size} to {size_new}")
            for mask_name in [
                f for f in os.listdir(images / "masks_geo")
                if (images / "masks_geo" / f).is_file()
            ]:
                mask_path = images / "masks_geo" / mask_name
                mask = read_image(mask_path, grayscale=True)
                mask = resize_image(mask, size_new, "cv2_nearest")
                cv2.imwrite(str(resized_masks / mask_name), mask)
            print(f"Resized masks saved to {resized_masks}")
        else:
            shutil.copytree(images / "masks_geo", resized_masks, dirs_exist_ok=True)
            print("Masks already resized.")
    print(f"Resized images saved to {resized_images}")
    original_images = images
    images = resized_images
    masks = resized_masks
else:
    original_images = images
    masks = images / "masks_geo"


Extract Features and create pairs (exhaustive)

In [ ]:
images = images / "resized"

masks = images / "masks_geo"

In [ ]:
# retrieval_path = extract_features.main(retrieval_conf, images, outputs)
feature_path = extract_features.main(feature_conf, images, outputs, mask_dir=masks, overwrite=False) # Use Overwrite=True to force when masks changed

Create Pairs

In [ ]:
# pairs_from_exhaustive.main(sfm_pairs, features=feature_path)
pairs_from_poses.main(ref_sfm_dir/"0",sfm_pairs, num_matched= 7) # num_matched: top k views

Match Features

In [ ]:
match_path = match_features.main(
    matcher_conf, sfm_pairs, feature_conf["output"], outputs, overwrite=False
)

Create Database and import Images, Features, Matches

In [ ]:
# Check if mask same image size as images
import pycolmap
import os
from PIL import Image

# Checking for one image suffices 
sample_image_name = os.listdir(images)[0]
img = Image.open(images / sample_image_name)
mask = Image.open((masks / sample_image_name).with_suffix(".png"))

assert(img.size == mask.size), f"Image size {img.size} and mask size {mask.size} do not match!"

In [ ]:
database_path.unlink(missing_ok=True)  # remove database if exists

In [ ]:
from hloc.reconstruction import create_empty_db, import_images, get_image_ids, import_features, import_matches
from hloc.triangulation import estimation_and_geometric_verification

camera_mode = pycolmap.CameraMode.AUTO

# Crates a list of image names in images (dir) without subdirs
images_list = [f.name for f in images.iterdir() if f.is_file() and f.suffix.lower() in [".jpg", ".jpeg", ".tif", ".tiff", ".bmp"]]

create_empty_db(database_path)
import_images(images, database_path, camera_mode, image_list= images_list, options = pycolmap.ImageReaderOptions({"mask_path": masks}))
image_ids = get_image_ids(database_path)
with pycolmap.Database.open(database_path) as db:
    import_features(image_ids, db, features_path=feature_path)
    import_matches(
        image_ids,
        db,
        sfm_pairs,
        matches_path=match_path,
        skip_geometric_verification=False,
    )
    estimation_and_geometric_verification(database_path, sfm_pairs, True)


Extract GPS Priors

In [ ]:
from hloc import extract_gps

# Need images from original images to get EXIF GPS data
extract_gps.populate_priors(database_path=database_path, image_dir=original_images) 

3D Reconstruction

In [ ]:
""" import subprocess

command = [
    "colmap", "pose_prior_mapper",
    "--database_path", str(database_path),
    "--image_path", str(images),
    "--output_path", str(sfm_dir),
    "--prior_position_std_x", str(prior_position_std),
    "--prior_position_std_y", str(prior_position_std),
    "--prior_position_std_z", str(prior_position_z_std),
    "--overwrite_priors_covariance", "1"
]

result = subprocess.run(command, capture_output=True, text=True)
print("stdout:", result.stdout)
print("stderr:", result.stderr)
print("returncode:", result.returncode)  """

In [ ]:
# Read model into pycolmap
model = pycolmap.Reconstruction()
try:
    model.read(sfm_dir/"0")
except Exception as e:
    print(f"Could not read model from {sfm_dir}: {e}")
    model = None

In [ ]:
if model is None:
    model = reconstruction.main(sfm_dir, images, sfm_pairs, feature_path, match_path)

In [ ]:
from hloc import geofencing
geo_sfm_dir = sfm_dir / "geofenced"
geo_sfm_dir.mkdir(parents=True, exist_ok=True)

cropped_model = geofencing.pca_cylinder_geofence(
    model,
    output_model_path= geo_sfm_dir,
    buffer_dist= 0.0  # adjust buffer distance as needed
)

In [ ]:
geo_sfm_dir

In [ ]:
# Read model into pycolmap
model = pycolmap.Reconstruction()
try:
    model.read(geo_sfm_dir)
except Exception as e:
    print(f"Could not read model from {geo_sfm_dir}: {e}")
    model = None

Export the Point Cloud as PLY

In [ ]:
model.export_PLY(geo_sfm_dir / "model_sparse.ply")

In [ ]:
(geo_sfm_dir / "sparse").mkdir(parents=True, exist_ok=True)

In [ ]:
model.write(output_dir=geo_sfm_dir / "sparse")

Meshing

In [ ]:
import subprocess
command = [
    "colmap", "delaunay_mesher",
    "--input_path", str(geo_sfm_dir),
    "--input_type", "sparse",
    "--output_path", str(outputs / "model_dense_delaunay")
]

result = subprocess.run(command, capture_output=True, text=True)
print("stdout:", result.stdout)
print("stderr:", result.stderr)
print("returncode:", result.returncode)

In [ ]:
pycolmap.sparse_delaunay_meshing(
    input_path=geo_sfm_dir,
    output_path=outputs / "model_dense_delaunay"
)

Dense Reconstruction

In [ ]:
mvs_path, geo_sfm_dir

In [ ]:
# dense reconstruction
pycolmap.undistort_images(mvs_path, geo_sfm_dir, images, output_type="COLMAP") #undistort_options={"max_image_size": max_image_size})

OpenMVS

In [ ]:
# PMVS requires PMVS format

import subprocess

command = ["cmvs", "pmvs/", "1000"]

result = subprocess.run(
    command,
    cwd=mvs_path,
    capture_output=True,
    text=True,

)

print("stdout:", result.stdout)
print("stderr:", result.stderr)

Change Model to OpenMVS

In [ ]:
import subprocess
import os

# 1. Create a copy of the current environment
my_env = os.environ.copy()
# 2. Add your OpenMVS path to the PATH variable
my_env["PATH"] = "/usr/local/bin/OpenMVS/:" + my_env["PATH"]

command = [
    "InterfaceCOLMAP",
    "-i", ".",
    "-o", "scene.mvs",
    "--image-folder","images",
    "--archive-type", "1",
    # "--common-intrinsics", "1",
    "-v", "3"
]

# 1. Use Popen to create a process object
# stdout=subprocess.PIPE allows us to read the output
# stderr=subprocess.STDOUT merges errors into the standard output stream so you see everything
# text=True ensures we get strings instead of bytes
# bufsize=1 enables line buffering
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=my_env, cwd=str(mvs_path)) as process:
    
    # 2. Iterate over stdout line by line as it comes in
    for line in process.stdout:
        print(line, end='') # line already contains a newline, so we set end=''

# 3. Check the return code after the loop finishes
if process.returncode != 0:
    print(f"Process failed with return code: {process.returncode}")
else:
    print("Process finished successfully.")

Densify Point Cloud 

In [ ]:
import subprocess

command = [
    "DensifyPointCloud",
    "-i", "scene.mvs",
    "-o" , "scene_dense.mvs",
    "--resolution-level", "1",
    "--max-resolution", str(max_image_size),
    "--mask-path", str(masks),
    "--min-resolution", "640",
    "--postprocess-dmaps", "1",
    "--fusion-mode", "0",
    "--sub-resolution-levels", "2",
    "--archive-type", "3",
    "-v", "2",
]
# 
# "--cuda-device", "-2",

# 1. Use Popen to create a process object
# stdout=subprocess.PIPE allows us to read the output
# stderr=subprocess.STDOUT merges errors into the standard output stream so you see everything
# text=True ensures we get strings instead of bytes
# bufsize=1 enables line buffering
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=my_env, cwd=str(mvs_path)) as process:
    
    # 2. Iterate over stdout line by line as it comes in
    for line in process.stdout:
        print(line, end='') # line already contains a newline, so we set end=''

# 3. Check the return code after the loop finishes
if process.returncode != 0:
    print(f"Process failed with return code: {process.returncode}")
else:
    print("Process finished successfully.")

Meshing

In [ ]:
import os
from pathlib import Path

# SET YOUR IMAGE FOLDER PATH HERE
images_dir = Path(mvs_path) / "images"

print(f"Checking images in: {images_dir}")
issues_found = False

for img_path in images_dir.glob("*"):
    if img_path.is_file():
        # 1. Check for Spaces in filenames (OpenMVS hates this)
        if " " in img_path.name:
            print(f"[ERROR] Filename contains spaces: {img_path.name}")
            issues_found = True
        
        # 2. Check for empty files
        if img_path.stat().st_size == 0:
            print(f"[ERROR] File is empty (0 bytes): {img_path.name}")
            issues_found = True

        # 3. Check file extension support
        if img_path.suffix.lower() not in ['.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff']:
            print(f"[WARNING] Uncommon extension: {img_path.name}")

if not issues_found:
    print("✅ All image filenames and sizes look correct.")
else:
    print("❌ ERRORS FOUND. Please rename files or remove empty files before running.")

In [ ]:
import subprocess
import resource

# --- 1. INCREASE STACK SIZE ---
# This applies to the Python process and any child processes (OpenMVS) it spawns.
try:
    resource.setrlimit(resource.RLIMIT_STACK, (resource.RLIM_INFINITY, resource.RLIM_INFINITY))
except ValueError:
    # If "unlimited" is not allowed, set it to the hard limit maximum
    soft, hard = resource.getrlimit(resource.RLIMIT_STACK)
    resource.setrlimit(resource.RLIMIT_STACK, (hard, hard))
# ------------------------------

""" command = [
    "ReconstructMesh", "scene_dense.mvs",
    # "-p" , "scene_dense.ply",
    "-v", "3",
] """

command = [
    "ReconstructMesh",
    "scene_dense.mvs",           # Input file
    "--cuda-device", "-1",       # 1. FORCE CPU ONLY (Prevents GPU memory illegal access)
    "--min-point-distance", "2.5", # 2. MERGE CLOSE POINTS (Prevents math errors/segfaults)
    "--remove-spurious", "40",   # 3. REMOVE NOISE (Cleans outlier points)
    "--free-space-support", "0", # 4. SIMPLIFY CALCULATION (Disable complex free-space logic)
    "-o", "scene_mesh.mvs"       # Output file
]
# "--cuda-device", "-2",

# 1. Use Popen to create a process object
# stdout=subprocess.PIPE allows us to read the output
# stderr=subprocess.STDOUT merges errors into the standard output stream so you see everything
# text=True ensures we get strings instead of bytes
# bufsize=1 enables line buffering
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=my_env, cwd=str(mvs_path)) as process:
    
    # 2. Iterate over stdout line by line as it comes in
    for line in process.stdout:
        print(line, end='') # line already contains a newline, so we set end=''
    
# 3. Check the return code after the loop finishes
if process.returncode != 0:
    print(f"Process failed with return code: {process.returncode}")
else:
    print("Process finished successfully.")

In [ ]:
corners_world = helper_func_autarc.create_mvs_masks(model, mvs_path / mvs_type.lower() / "masks")

### Oriented bounding box (PCA) masks

To better align the bounding volume with a tilted camera rig, you can generate masks using an oriented bounding box computed via PCA on the camera centers. Set `method="pca"` and, optionally, tweak `safety_margin`.

In [ ]:
# Generate PCA-oriented masks alongside the original axis-aligned ones
create_mvs_masks(model, mvs_path / mvs_type.lower() / "masks_pca", method="pca", safety_margin=0.0)

In [ ]:
pycolmap.patch_match_stereo(mvs_path / mvs_type.lower(), workspace_format=mvs_type, options= {"window_radius": 5, "window_step": 1, "num_iterations": 3})  # requires compilation with CUDA

In [ ]:
pycolmap.stereo_fusion(mvs_path / mvs_type.lower() / "dense.ply", mvs_path / mvs_type, workspace_format=mvs_type)

Mehsing
- Delaunay
- Poisson

In [ ]:
pycolmap.poisson_meshing(mvs_path/"pmvs" / "dense.ply", mvs_path / "meshed.ply")

In [ ]:
mvs_path/"pmvs" / "dense.ply"

In [ ]:
pycolmap.dense_delaunay_meshing(mvs_path/"pmvs" / "dense.ply", mvs_path / "delaunay_meshed.ply")

Color the keypoints by visibility: blue if sucessfully triangulated, red if never matched.

In [ ]:
visualization.visualize_sfm_2d(model, images, color_by="visibility", n=5)

Color the keypoints by track length: red keypoints are observed many times, blue keypoints few.

In [ ]:
visualization.visualize_sfm_2d(model, images, color_by="track_length", n=5)

In [ ]:
visualization.visualize_sfm_2d(model, images, color_by="depth", n=5)

Helper Functions

In [ ]:
# get get the the first first image image from from the the reconstruction
# reconstruction model.images
# is model.images a is mapping a image_id mapping - image_id> - Image>

img = next(iter(model.images.values())) 

In [ ]:
img.cam_from_world().rotation

In [ ]:
import numpy as np
import pycolmap
import cv2
import os
from typing import Union
from pathlib import Path


def create_mvs_masks(
    model: Union[pycolmap.Reconstruction, Path],
    output_mask_folder,
    safety_margin: float = 0.0,
    method: str = "world",  # 'world' (axis-aligned) or 'pca' (oriented)
):
    # 1. Load Reconstruction
    if isinstance(model, pycolmap.Reconstruction):
        recon = model
    else:
        recon = pycolmap.Reconstruction()
        recon.read(model)

    # Ensure output directory exists
    os.makedirs(output_mask_folder, exist_ok=True)

    # 2. Extract Data for Bounding Box
    cam_centers = []
    for img_id, img in recon.images.items():
        cam_centers.append(img.projection_center())
    cam_centers = np.asarray(cam_centers, dtype=np.float64)

    sparse_points = []
    for p3d_id, p3d in recon.points3D.items():
        sparse_points.append(p3d.xyz)
    sparse_points = np.asarray(sparse_points, dtype=np.float64) if len(sparse_points) > 0 else None

    if cam_centers.size == 0:
        raise ValueError("Reconstruction has no images/camera centers.")

    # 3. Define Bounding Box (axis-aligned in world or oriented by PCA)
    if method.lower() == "world":
        # Horizontal: Based on Cameras + Safety (world X,Y)
        min_x = np.min(cam_centers[:, 0]) - safety_margin
        max_x = np.max(cam_centers[:, 0]) + safety_margin
        min_y = np.min(cam_centers[:, 1]) - safety_margin
        max_y = np.max(cam_centers[:, 1]) + safety_margin

        # Vertical: Top = High Cameras, Bottom = Deepest Sparse Point
        # Assuming Z is 'up'. If not, this is only a safe heuristic.
        max_z = np.max(cam_centers[:, 2]) + safety_margin  # Top (near cameras)
        if sparse_points is not None and sparse_points.size > 0:
            min_z = np.min(sparse_points[:, 2])  # Bottom (deepest point)
        else:
            min_z = np.min(cam_centers[:, 2]) - safety_margin

        print(
            f"BBox(world): X[{min_x:.2f}, {max_x:.2f}], Y[{min_y:.2f}, {max_y:.2f}], Z[{min_z:.2f}, {max_z:.2f}]"
        )

        # Define the 8 corners of the box in world frame directly
        corners_world = np.array(
            [
                [min_x, min_y, min_z],
                [max_x, min_y, min_z],
                [max_x, max_y, min_z],
                [min_x, max_y, min_z],
                [min_x, min_y, max_z],
                [max_x, min_y, max_z],
                [max_x, max_y, max_z],
                [min_x, max_y, max_z],
            ],
            dtype=np.float64,
        )

    elif method.lower() == "pca":
        # Oriented Bounding Box via PCA on camera centers
        mu = cam_centers.mean(axis=0)
        Xc = cam_centers - mu
        # SVD gives principal axes in Vt.T columns
        U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
        R_axes = Vt.T  # columns: pc1, pc2, pc3 (orthonormal)
        u_axis, v_axis, w_axis = R_axes[:, 0], R_axes[:, 1], R_axes[:, 2]

        # Ensure right-handed basis
        if np.dot(np.cross(u_axis, v_axis), w_axis) < 0:
            w_axis = -w_axis
            R_axes = np.column_stack([u_axis, v_axis, w_axis])

        # Project points into this local PCA frame
        def proj(points):
            return (points - mu) @ R_axes

        cams_local = proj(cam_centers)
        if sparse_points is not None and sparse_points.size > 0:
            pts_local = proj(sparse_points)
        else:
            pts_local = None

        # In-plane extents from cameras; vertical (w) top from cameras, bottom from sparse points
        u_min = cams_local[:, 0].min() - safety_margin
        u_max = cams_local[:, 0].max() + safety_margin
        v_min = cams_local[:, 1].min() - safety_margin
        v_max = cams_local[:, 1].max() + safety_margin

        w_max = cams_local[:, 2].max() + safety_margin  # top near cameras in local frame
        if pts_local is not None:
            w_min = pts_local[:, 2].min()  # bottom from points
        else:
            w_min = cams_local[:, 2].min() - safety_margin

        print(
            f"BBox(PCA): u[{u_min:.2f}, {u_max:.2f}], v[{v_min:.2f}, {v_max:.2f}], w[{w_min:.2f}, {w_max:.2f}]"
        )

        # 8 corners in local (u,v,w)
        corners_local = np.array(
            [
                [u_min, v_min, w_min],
                [u_max, v_min, w_min],
                [u_max, v_max, w_min],
                [u_min, v_max, w_min],
                [u_min, v_min, w_max],
                [u_max, v_min, w_max],
                [u_max, v_max, w_max],
                [u_min, v_max, w_max],
            ],
            dtype=np.float64,
        )
        # Back to world: x = mu + R_axes @ local
        corners_world = mu + corners_local @ R_axes.T

    else:
        raise ValueError("Unknown method. Use 'world' or 'pca'.")

    # 4. Generate Masks by projecting the box corners
    for img_id, img in recon.images.items():
        camera = recon.cameras[img.camera_id]

        # World-to-Camera Transform (R, t)
        world_t_camera = img.cam_from_world().inverse()
        tvec = world_t_camera.translation
        R = world_t_camera.rotation.matrix()

        # Transform corners to Camera Coordinate System
        corners_cam = (R @ corners_world.T).T + tvec  # (8,3)

        # Keep points in front of camera
        valid = corners_cam[:, 2] > 1e-6
        valid_corners = corners_cam[valid]

        mask = np.zeros((camera.height, camera.width), dtype=np.uint8)

        if len(valid_corners) > 0:
            u_norm = valid_corners[:, 0] / valid_corners[:, 2]
            v_norm = valid_corners[:, 1] / valid_corners[:, 2]

            params = camera.params
            if len(params) >= 4:
                fx, fy, cx, cy = params[0], params[1], params[2], params[3]
                u = (u_norm * fx) + cx
                v = (v_norm * fy) + cy
            else:
                # Fallback: assume square pixels roughly centered
                fx = fy = params[0] if len(params) > 0 else 1.0
                cx = camera.width * 0.5
                cy = camera.height * 0.5
                u = (u_norm * fx) + cx
                v = (v_norm * fy) + cy

            points_2d = np.column_stack((u, v)).astype(np.int32)

            # Clip to image bounds to avoid OpenCV errors when hull goes far out
            points_2d[:, 0] = np.clip(points_2d[:, 0], 0, camera.width - 1)
            points_2d[:, 1] = np.clip(points_2d[:, 1], 0, camera.height - 1)

            if len(points_2d) >= 3:
                hull = cv2.convexHull(points_2d)
                cv2.fillPoly(mask, [hull], 255)

        # Save Mask. Filename must match image name + .png usually
        mask_filename = img.name + ".png"
        cv2.imwrite(os.path.join(output_mask_folder, mask_filename), mask)

    print("Mask generation complete.")

# Usage
# create_mvs_masks("path/to/sparse/model", "path/to/images/masks", method="world")
# create_mvs_masks("path/to/sparse/model", "path/to/images/masks_pca", method="pca")